In [ ]:
# A lot fo the things we can do in pandas, we could do using SQLAlchamy before bringing the data in, the same as we would when using SQL normally.
# This keeps the amount of data we are bringing in smaller and easier to work with. However, this course is time bound and I don't want to spend too
# long on SQL things when not everyone has access to it. Instead we'll look at some basics but do most processing in pandas. I have further notes
# at the two following links for more in depth SQLAlchemy including doing calculations, making more complex selections, applying functions, and joining:
# https://github.com/data-to-insight/ERN-sessions/blob/main/SQL%20in%20python/session_1.ipynb https://github.com/data-to-insight/ERN-sessions/blob/main/SQL%20in%20python/session_2.ipynb

# I'd really recommend google, however. Also, as you'll see, if you already know SQL you can simply write the queries you want inside SQLAlchemy so you
#  can already do the queries first, AND you don't need to learn anything.

# To install the sqlalchemy package, run the code below in the terminal
# pip install sqlalchemy

from sqlalchemy import (create_engine, 
                        inspect, 
                        text, 
                        select, 
                        MetaData, 
                        Table, 
                        and_,
                        or_,
                        desc,
                        asc,
                        func,
                        )

# we need to use //// in this instance as we are giving a relative absolute path to where our DB file is
# Also remember we can't start a variable name with a number
# engine_903 = create_engine("sqlite+pysqlite:////workspaces/ERN-sessions/Intermediate Python/903_database.db") #, echo=True)

# everyone's path may be slightly different due to file structure
engine_903 = create_engine("sqlite+pysqlite:////workspaces/python_learning/Python_tutorial_code/Intermediate/data/903_database.db")
# We cans et echo=True if we want to see what our DB connections are doing

connection = engine_903.connect()
inspection = inspect(engine_903)
inspection.get_table_names()

In [ ]:
engine_gravity = create_engine("sqlite+pysqlite://///workspaces/python_learning/Python_tutorial_code/Intermediate/data/gravity.db")
# We cans et echo=True if we want to see what our DB connections are doing

connection = engine_gravity.connect()
inspection = inspect(engine_gravity)
inspection.get_table_names()

In [ ]:
# This is called 'Reflecting' and is used to reflect data from an already existing db, rather than make it,
# it takes information about the data in the table from the DB (metadata) and uses it to make tables
# to reflect a table initialise a MetaData object,
# This is a bit like making an empty table so you can put data in it later, and it'll put the right stuff in!

# https://docs.sqlalchemy.org/en/20/core/reflection.html

metadata_903 = MetaData() 

# if we print it we can see it's empty!
print(metadata_903.tables.keys())

episodes = Table('episodes', metadata_903, autoload_with=engine_903)
print(episodes.name) # prints table name
print(episodes.c.keys()) # prints column names
print(repr(episodes)) # repr() function lets us view the details of our table, like .info() for a df

In [ ]:
# Now do the same for the gravity db
gravity_metadata = MetaData() 

print(gravity_metadata.tables.keys())

books = Table('book', gravity_metadata, autoload_with=engine_gravity)
print(books.name)
print(books.c.keys())
print(repr(books))

In [ ]:
# In Python with is a bit like a for or an if, it does stuff in indented blocks, it lets you temporarily initialise a variable
# it's also really good for if you have a resource you don't want to use all the time (like a db connection or file stream),
# because it'll only be being used during the block

with engine_903.connect() as con:
    stmt = "SELECT * FROM episodes" # normal sql theory for querying, it's easier to do it in a Python way, as seen later
    result_proxy = con.execute(text(stmt)) # this is done to say how much data we want
    results = result_proxy.fetchall() # contains actual data
    
first_row = results[0]
print(first_row) # prints first row
print(first_row.CHILD) # prints CHILD column of first row

In [ ]:
# Find the first row of the book table, print it, then print the value of the 'title'

with engine_gravity.connect() as con:
    stmt = "SELECT * FROM book"
    result_proxy = con.execute(text(stmt))
    results = result_proxy.fetchall()
first_row = results[0]
print(first_row)
print(first_row.title) 

In [ ]:
with engine_gravity.connect() as con:
    stmt = "SELECT * FROM book"
    result_proxy = con.execute(text(stmt))
    results = result_proxy.fetchmany(size=10)

print(results) 

In [ ]:
# we can also use Python to make selections, rather than SQL 
# we use the reflection of the tracks table we made earlier

with engine_903.connect() as con:
    stmt = select(episodes) # Pythonic select statement
    print(stmt) # See how it converts it to SQL
    result_proxy = con.execute(stmt) # this is done to say how much data we want
    results = result_proxy.fetchmany(size=10) # fetches first ten results
print(results)

In [ ]:
with engine_gravity.connect() as con:
    stmt = select(books) # Pythonic select statement
    print(stmt) # See how it converts it to SQL
    result_proxy = con.execute(stmt) # this is done to say how much data we want
    results = result_proxy.fetchmany(size=10) # fetches first ten results
print(results)

In [ ]:
# We can make more complex queries too, like we would using SQL or pandas.
# note to differentiate these conjuenctions form the build in Python ones 
# they finish with an underscore. If they didn't, Pyhton would be trying to
# use the build in ones to a weird effect. If ever I talk about namespace, this is one reason why.

from sqlalchemy import or_
header = Table('header', metadata_903, autoload_with=engine_903)

with engine_903.connect() as con:
    # Selecting all tracks where the album ID column is 1 or 2 (Third column)
    stmt = select(header).where(or_(header.columns.ETHNIC == "REFU", 
                                      header.columns.ETHNIC == "NOBT")) 
    result_proxy = con.execute(stmt)
    results = result_proxy.fetchmany(size=20)
print(results)

In [ ]:
# select all of the book table where the book_id is either 1 or 2

# two rows below not required?!?
#from sqlalchemy import or_
#header = Table('header', metadata_903, autoload_with=engine_gravity)

with engine_gravity.connect() as con:
    # Selecting all tracks where the album ID column is 1 or 2 (Third column)
    stmt = select(books).where(or_(books.columns.book_id == 1, 
                                      books.columns.book_id == 2)) 
    result_proxy = con.execute(stmt)
    results = result_proxy.fetchmany(size=10)
print(results)

In [ ]:
# We can easily convert results to dataframes by just passing the result to a dataframe class
# pip install pandas

import pandas as pd

ad1 = Table('ad1', metadata_903, autoload_with=engine_903)
with engine_903.connect() as con:
    stmt = select(ad1)
    result = con.execute(stmt).fetchall()
df = pd.DataFrame(result)

In [ ]:
df

In [ ]:
with engine_gravity.connect() as con:
    statement = select(books)
    result_proxy = con.execute(statement)
    results = result_proxy.fetchall()

# print(results)

df = pd.DataFrame(results)

df